# Track 4 · 03 VEMG2Pose — Meta's pretrained decoder: zero-shot, then fine-tuned

**One job.** Score Meta's released emg2pose checkpoint on our 15 unseen validation
people as it is, fine-tune it on our balanced sample, score it again, and package the
better of the two.

Self-contained: upload this one file. The windows come from `track4_01_prepare` on Drive.

---

## The model

braindecode's `VEMG2Pose` (Salter et al. 2024), the architecture behind NeuralBench's
17.5° reference: a time-depth-separable convolutional encoder on raw 2 kHz EMG, then a
two-layer LSTM that rolls the pose out at 50 Hz. Weights: Meta's released regression
checkpoint, `braindecode/VEMG2Pose-emg2pose` (5.98M parameters, **CC-BY-NC-SA-4.0**,
declare it in the submission). It was trained on emg2pose's training split, so it has not
seen our 15 validation people, the 20 scored test people, or the 6 held-out movement
types; our sample includes those movement types (from training-split people), which
fine-tuning can add.

Checked before building this notebook, on real windows: the checkpoint expects
**microvolts** (best of scales 1e4 to 1e7), and our 1 kHz storage upsampled back to 2 kHz
gives the same error as native data (13.83° vs 13.84°), so the existing sample serves as
is. The model needs 1,790 samples (0.9 s) of lead-in before its first prediction; the
harness gives none, so each window's start is mirrored in front of it, here and in the
submission alike.

## Scoring

Mean absolute angle error in degrees on the 15 validation people, the same yardstick as
`track4_02`: EEGNet reached 17.64° there (18.24° on the warm-up board), the mean pose
19.04°.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

FINE_TUNE = True
EPOCHS = 8
LR = 1e-4                 # small: start from the pretrained solution, do not overwrite it
LR_SCHEDULE = 'cosine'
BATCH = 32
ROTATE = True             # task config's band rotation, offsets -1/0/+1
WEIGHT_DECAY = 0.0
SEED = 33
MAKE_ZIP = True

EEGNET_VAL = 17.64        # track4_02, F1 32: val 17.64 deg, warm-up board 18.24
CHECKPOINT = 'braindecode/VEMG2Pose-emg2pose'
RUN_TAG = f'ft{int(FINE_TUNE)}_e{EPOCHS}_lr{LR:g}_b{BATCH}_rot{int(ROTATE)}'
print(f'NOTEBOOK VERSION: track4_03 v1   settings {RUN_TAG}')

In [ ]:
%pip install -q 'braindecode>=1.8'

import importlib
for mod in ('braindecode', 'torch', 'numpy'):
    print(f'{mod:12} {importlib.import_module(mod).__version__}')

In [ ]:
import copy, json, time, warnings, zipfile
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F
from braindecode.models import VEMG2Pose
from braindecode.augmentation.functional import band_rotation

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

BASE = Path(ROOT) / 'Track4'
DATA = BASE / 'emg2pose'
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


def load_npz(path, tries=5):
    """np.load that survives the Drive mount dropping mid-read."""
    for attempt in range(tries):
        try:
            with np.load(path) as z:
                return {k: z[k] for k in z.files}
        # A dropped mount surfaces as OSError, or as BadZipFile when it hits zipfile's
        # first read of the archive.
        except (OSError, zipfile.BadZipFile, EOFError) as e:
            if attempt == tries - 1 or not ROOT.startswith('/content/drive'):
                raise
            print(f'Drive read failed on {Path(path).name} ({e}); remounting and retrying')
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)


man = json.loads((DATA / 'manifest.json').read_text())
ST = man['stored']
parts = {g: ([], [], []) for g in ('train', 'val')}
t0 = time.time()
for f in sorted((DATA / 'windows').glob('sub-*.npz')):
    z = load_npz(f)
    if 'X' not in z:
        continue
    info = json.loads(str(z['info']))
    p = parts[info['group']]
    p[0].append(z['X']); p[1].append(z['Y']); p[2].append(np.full(len(z['X']), info['sub']))
X, Y, P = {}, {}, {}
for g, (xs, ys, ps) in parts.items():
    X[g], Y[g], P[g] = np.concatenate(xs), np.concatenate(ys), np.concatenate(ps)
assert not set(P['train']) & set(P['val'])
N_CHANS, N_STORED = X['train'].shape[1:]
N_JOINTS, OUT_STEPS = Y['train'].shape[1:]
NATIVE = man['harness']['native_sfreq']
N_NATIVE = int(round(N_STORED * NATIVE / ST['emg_sfreq']))
STEP = N_NATIVE // OUT_STEPS                  # model output samples per stored target step
print(f'train {len(X["train"]):,} windows / {len(set(P["train"]))} people, val {len(X["val"]):,} / '
      f'{len(set(P["val"]))} people   ({time.time() - t0:.0f}s), device {DEVICE}')

deg = lambda r: float(np.degrees(r))
mean_pose = Y['train'].astype('float32').mean(axis=(0, 2), keepdims=True)
floor_val = deg(np.abs(np.broadcast_to(mean_pose, Y['val'].shape) - Y['val'].astype('float32')).mean())
print(f'mean pose on val: {floor_val:.2f} deg')

## Step 1 — the pretrained model, and the input pipeline the submission repeats

Stored windows are 1 kHz microvolts with each window's mean removed. They are upsampled
to the model's 2 kHz with the same resampler, given their mirrored lead-in, and the
output is read at the stored targets' 200 Hz (every tenth sample, the harness's own
nearest rule).

In [ ]:
def resample_fft(x, n_out):
    """Same function as the prepare notebook and the submission."""
    n_in = x.shape[-1]
    if n_in == n_out:
        return x
    spec = torch.fft.rfft(x.float(), dim=-1)
    out = torch.zeros(*spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device)
    keep = min(spec.shape[-1], out.shape[-1])
    out[..., :keep] = spec[..., :keep]
    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)


def lead_in(x, n):
    """Same as the submission: mirror the first n samples in front of the window."""
    return torch.cat([x[..., 1:n + 1].flip(-1), x], dim=-1)


def model_input(xb):
    """Stored windows (B, C, 5000) float16 microvolts -> model input at 2 kHz."""
    return lead_in(resample_fft(xb.to(DEVICE).float(), N_NATIVE), net.left_context)


def forward(model, xb):
    """(B, J, OUT_STEPS) radians, read off the model's 2 kHz output."""
    return model(model_input(xb)).transpose(1, 2)[..., ::STEP]


@torch.no_grad()
def evaluate(model, Xa, Ya, Pa):
    model.eval()
    Xa = torch.as_tensor(Xa)
    pred = torch.cat([forward(model, Xa[i:i + 64]).cpu() for i in range(0, len(Xa), 64)]).numpy()
    err = np.abs(pred - Ya.astype('float32'))
    per_person = {s: deg(err[Pa == s].mean()) for s in sorted(set(Pa))}
    return deg(err.mean()), per_person, np.degrees(err.mean(axis=(0, 2))), pred


net = VEMG2Pose.from_pretrained(CHECKPOINT).to(DEVICE)
print(f'{CHECKPOINT}: {sum(p.numel() for p in net.parameters()):,} parameters, '
      f'left context {net.left_context} samples, rate {net.input_sfreq:g} Hz')

## Step 2 — zero-shot: the checkpoint as released

In [ ]:
t0 = time.time()
zs_mae, zs_person, zs_joint, zs_pred = evaluate(net, X['val'], Y['val'], P['val'])
pp = np.array(list(zs_person.values()))
print(f'ZERO-SHOT  val MAE {zs_mae:.2f} deg   ({time.time() - t0:.0f}s)')
print(f'  per person: best {pp.min():.2f}  median {np.median(pp):.2f}  worst {pp.max():.2f}')
print(f'  per joint: easiest {zs_joint.min():.1f}  hardest {zs_joint.max():.1f} deg')
print(f'  EEGNet (track4_02) {EEGNET_VAL:.2f}   mean pose {floor_val:.2f}   '
      f'-> zero-shot is {EEGNET_VAL - zs_mae:+.2f} deg better than EEGNet')

## Step 3 — fine-tune on our sample

Small learning rate, L1 loss on the angles, the task's band rotation, and the best epoch
on the validation people kept. Epoch 0 is the zero-shot model, so fine-tuning can only
replace it by beating it.

In [ ]:
best = dict(epoch=0, val=zs_mae, state=copy.deepcopy(net.state_dict()))
hist = []
if FINE_TUNE:
    gen = torch.Generator().manual_seed(SEED)
    aug_rng = np.random.RandomState(SEED)
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
             if LR_SCHEDULE == 'cosine' else None)
    Xtr, Ytr = torch.from_numpy(X['train']), torch.from_numpy(Y['train'])
    for ep in range(1, EPOCHS + 1):
        net.train()
        order = torch.randperm(len(Xtr), generator=gen)
        tot, t0 = 0.0, time.time()
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            xb, yb = Xtr[b].to(DEVICE).float(), Ytr[b].to(DEVICE).float()
            if ROTATE:
                xb, yb = band_rotation(xb, yb, num_bands=1, electrodes_per_band=N_CHANS,
                                       band_offsets=(-1, 0, 1), random_state=aug_rng)
            opt.zero_grad()
            loss = F.l1_loss(forward(net, xb), yb)
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        lr_now = opt.param_groups[0]['lr']
        if sched is not None:
            sched.step()
        v, _, _, _ = evaluate(net, X['val'], Y['val'], P['val'])
        hist.append(dict(epoch=ep, train_l1_deg=deg(tot / len(order)), val_mae_deg=v))
        better = v < best['val']
        if better:
            best = dict(epoch=ep, val=v, state=copy.deepcopy(net.state_dict()))
        print(f'  epoch {ep:2d}  train L1 {deg(tot / len(order)):6.2f} deg  val MAE {v:6.2f} deg  '
              f'lr {lr_now:.1e}{"  *" if better else ""}  {time.time() - t0:.0f}s')

net.load_state_dict(best['state'])
ft_mae, ft_person, ft_joint, ft_pred = evaluate(net, X['val'], Y['val'], P['val'])
null = np.array([deg(np.abs(ft_pred[rng.permutation(len(ft_pred))] - Y['val'].astype('float32')).mean())
                 for _ in range(30)])
pp = np.array(list(ft_person.values()))
print(f'\n{"FINE-TUNED" if best["epoch"] else "KEPT ZERO-SHOT"}  val MAE {ft_mae:.2f} deg  '
      f'(epoch {best["epoch"]}; 0 = zero-shot)')
print(f'  zero-shot {zs_mae:.2f}   EEGNet {EEGNET_VAL:.2f}   mean pose {floor_val:.2f}   '
      f'null {null.mean():.2f} +- {null.std():.2f}')
print(f'  per person: best {pp.min():.2f}  median {np.median(pp):.2f}  worst {pp.max():.2f}')
print('  warm-up references: zeros 22.67, mean pose 21.60, NeuroPose 17.5, our EEGNet 18.24 (deg)')
TRENDING = bool(ft_mae < null.min() and ft_mae < floor_val)

torch.save(net.state_dict(), WEIGHTS / f'vemg2pose_{RUN_TAG}.pt')
(WEIGHTS / f'vemg2pose_{RUN_TAG}.json').write_text(json.dumps(dict(
    run_tag=RUN_TAG, checkpoint=CHECKPOINT, zero_shot_val_deg=zs_mae, kept_epoch=best['epoch'],
    val_mae_deg=ft_mae, per_person=ft_person, history=hist), indent=1))
print(f'\nsaved -> {WEIGHTS / f"vemg2pose_{RUN_TAG}.pt"}')

## Step 4 — package for submission

The weights travel in the zip, so nothing is downloaded at scoring. The checks feed the
submission raw-style 2 kHz volts with a large offset and compare it with this notebook's
own pipeline on the same signal.

In [ ]:
# Source of truth is solvers/track4_pose_vemg2pose.py in the repo.
SUBMISSION_PY = "\"\"\"Track 04 submission: VEMG2Pose, Meta's pretrained emg2pose decoder, optionally fine-tuned.\n\nThe model is braindecode's `VEMG2Pose` (Salter et al. 2024): a time-depth-separable\nconvolutional encoder feeding an LSTM that rolls the hand pose out at 50 Hz. Its\nweights start from Meta's released regression checkpoint\n(`braindecode/VEMG2Pose-emg2pose`, CC-BY-NC-SA-4.0) and may be fine-tuned in\n`track4_03_vemg2pose.ipynb`. They ship in this zip; nothing is downloaded at scoring.\n\nInput handling repeats what the model saw in training: per-window, per-channel mean\nremoval, a round trip through 1 kHz with the same rfft resampler (the training windows\nwere stored at 1 kHz), and volts to microvolts, the scale the checkpoint expects. The\nmodel consumes 1,790 samples of lead-in before its first prediction and the harness\ngives none, so the window's first 1,790 samples are mirrored in front of it. Electrodes\nare matched by name; output is (B, 20, T) radians at the input rate.\n\"\"\"\n\nimport json\n\nimport torch\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import VEMG2Pose\n\n\ndef resample_fft(x, n_out):\n    \"\"\"Resample along time via the rfft spectrum; the prepare notebook uses the same.\"\"\"\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\ndef lead_in(x, n):\n    \"\"\"Mirror the first n samples in front of the window, for the model's left context.\"\"\"\n    return torch.cat([x[..., 1:n + 1].flip(-1), x], dim=-1)\n\n\nclass ChannelMatchedVEMG2Pose:\n\n    def __init__(self, cfg, in_names, in_sfreq, n_joints, device):\n        if n_joints not in (None, cfg[\"n_joints\"]):\n            raise RuntimeError(f\"trained for {cfg['n_joints']} joints, harness asks for {n_joints}\")\n        self.cfg, self.device = cfg, device\n        self.in_sfreq = float(in_sfreq)\n        self.net = VEMG2Pose(\n            n_outputs=cfg[\"n_joints\"], n_chans=len(cfg[\"channels\"]),\n            sfreq=cfg[\"model_sfreq\"], n_times=cfg[\"model_n_times\"],\n        ).to(device).eval()\n        lookup = {name.casefold(): i for i, name in enumerate(in_names)}\n        self.src = [lookup.get(name.casefold(), -1) for name in cfg[\"channels\"]]\n        missing = [n for n, i in zip(cfg[\"channels\"], self.src) if i < 0]\n        print(\n            f\"[VEMG2Pose] matched {len(self.src) - len(missing)}/{len(self.src)} \"\n            f\"training channels by name among {len(in_names)} inputs; \"\n            f\"zero-filled: {missing if missing else 'none'}\"\n        )\n        if len(missing) == len(self.src):\n            raise RuntimeError(f\"no training channel found in the input: {list(in_names)[:8]}\")\n\n    @torch.no_grad()\n    def predict(self, X):\n        cfg = self.cfg\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        x = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                x[:, dst] = X[:, src]\n        x = x - x.mean(-1, keepdim=True)\n        n_model = int(round(x.shape[-1] * cfg[\"model_sfreq\"] / self.in_sfreq))\n        n_stored = int(round(x.shape[-1] * cfg[\"stored_sfreq\"] / self.in_sfreq))\n        x = resample_fft(resample_fft(x, n_stored), n_model) * cfg[\"emg_scale\"]\n        out = self.net(lead_in(x, self.net.left_context))          # (B, T, J)\n        return out.transpose(1, 2)                                  # (B, J, T) radians\n\n\nclass Solver(CompetSolver):\n\n    name = \"VEMG2Pose\"\n\n    requirements = [\"pip::braindecode\"]\n\n    def load_model(self, meta):\n        sub = meta[\"submission_dir\"]\n        cfg = json.loads((sub / \"vemg2pose_config.json\").read_text())\n        in_names = meta.get(\"ch_names\") or [c[\"ch_name\"] for c in meta[\"chs_info\"]]\n        model = ChannelMatchedVEMG2Pose(cfg, in_names, meta[\"sfreq\"], meta.get(\"n_joints\"),\n                                        meta[\"device\"])\n        model.net.load_state_dict(torch.load(sub / \"weights.pt\", map_location=meta[\"device\"],\n                                             weights_only=True))\n        return model\n"

In [ ]:
if MAKE_ZIP and not (TRENDING or globals().get('FORCE_ZIP')):
    print('Not packaging: the model did not clear the mean pose and its null. Set FORCE_ZIP = True to override.')
elif MAKE_ZIP:
    import importlib.util, shutil, sys, types
    out = BASE / 'submissions' / f'track4_vemg2pose_{RUN_TAG}'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    torch.save({k: v.cpu() for k, v in net.state_dict().items()}, out / 'weights.pt')
    CH = [f'emg{i}' for i in range(N_CHANS)]
    (out / 'vemg2pose_config.json').write_text(json.dumps(dict(
        channels=CH, n_joints=int(N_JOINTS), model_sfreq=float(net.input_sfreq),
        model_n_times=int(net.n_times), stored_sfreq=ST['emg_sfreq'], emg_scale=ST['emg_scale'],
        checkpoint=CHECKPOINT, licence='CC-BY-NC-SA-4.0', fine_tuned_epochs=best['epoch'],
        trained_on=f'emg2pose balanced sample, {len(set(P["train"]))} people (scored test people excluded)',
        val_mae_deg=ft_mae), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY)

    stub = types.ModuleType('benchmark_utils.base_solver')
    stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
    sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
    sys.modules['benchmark_utils.base_solver'] = stub
    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    def run(names, Xa, n_joints=N_JOINTS):
        m = mod.Solver().load_model({'ch_names': list(names), 'chs_info': None, 'sfreq': NATIVE,
                                     'submission_dir': out, 'device': DEVICE, 'n_joints': n_joints})
        return m.predict(torch.tensor(Xa)).cpu().numpy()

    raw_x = (rng.standard_normal((4, N_CHANS, N_NATIVE)) * 3e-5 + 0.05).astype('float32')
    stored = resample_fft(torch.from_numpy(raw_x - raw_x.mean(-1, keepdims=True)), N_STORED) * ST['emg_scale']
    with torch.no_grad():
        net.eval()
        expect = net(model_input(stored)).transpose(1, 2).cpu().numpy()
    base = run(CH, raw_x)
    perm = rng.permutation(N_CHANS)
    shuffled = run([CH[i] for i in perm], raw_x[:, perm])
    no_offset = run(CH, raw_x - 0.05)
    try:
        run(CH, raw_x[:1], n_joints=21); wrong = False
    except RuntimeError:
        wrong = True
    checks = [
        ('shape and range', base.shape == (4, N_JOINTS, N_NATIVE) and np.isfinite(base).all()
         and np.abs(base).max() < 2 * np.pi, f'{base.shape}, |angle| <= {np.abs(base).max():.2f} rad'),
        ('same pipeline as training', np.allclose(base, expect, atol=1e-3),
         f'max |diff| {np.abs(base - expect).max():.1e} rad'),
        ('offset removed', np.allclose(base, no_offset, atol=1e-3), 'a 50 mV offset changes nothing'),
        ('order-independent', np.allclose(base, shuffled, atol=1e-4), 'electrodes shuffled'),
        ('wrong joint count raises', wrong, 'asked for 21 joints: refuses rather than guessing'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if all(c[1] for c in checks):
        zpath = shutil.make_archive(str(out), 'zip', out)
        print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e6:.0f} MB)')
        print('Upload to Codabench (Track 4). Declare the model: VEMG2Pose, Meta emg2pose '
              'checkpoint (braindecode/VEMG2Pose-emg2pose, CC-BY-NC-SA-4.0)'
              + (', fine-tuned on emg2pose.' if best['epoch'] else '.'))
    else:
        print('\nNot zipping: a check failed.')